# 获取初代 151 只宝可梦数据

下面的代码会从 PokeAPI 抓取 1~151 号宝可梦的数据，并保存到 `../data/pokemon151.json`。

In [1]:
import json
import time
import urllib.error
import urllib.request
from pathlib import Path

POKEAPI_BASE = "https://pokeapi.co/api/v2/pokemon"
OUTPUT_FILE = Path("../data/pokemon151.json").resolve()
START_ID = 1
END_ID = 151
RETRY_DELAY = 2
MAX_RETRIES = 3


def fetch_pokemon(pokemon_id: int) -> dict:
    url = f"{POKEAPI_BASE}/{pokemon_id}"
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with urllib.request.urlopen(url, timeout=10) as response:
                return json.load(response)
        except urllib.error.HTTPError as e:
            print(f"HTTPError fetching {pokemon_id}: {e.code} {e.reason}")
            if 500 <= e.code < 600 and attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
                continue
            raise
        except urllib.error.URLError as e:
            print(f"URLError fetching {pokemon_id}: {e.reason}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
                continue
            raise


pokemon_list = []
for pokemon_id in range(START_ID, END_ID + 1):
    print(f"Fetching Pokémon {pokemon_id}...")
    pokemon_data = fetch_pokemon(pokemon_id)
    pokemon_list.append(pokemon_data)
    time.sleep(0.2)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_FILE, "w", encoding="utf-8") as fp:
    json.dump(pokemon_list, fp, ensure_ascii=False, indent=2)

print(f"完成：已保存 {len(pokemon_list)} 只宝可梦到 {OUTPUT_FILE}")

Fetching Pokémon 1...
Fetching Pokémon 2...
Fetching Pokémon 3...
Fetching Pokémon 4...
Fetching Pokémon 5...
Fetching Pokémon 6...
Fetching Pokémon 7...
Fetching Pokémon 8...
Fetching Pokémon 9...
Fetching Pokémon 10...
Fetching Pokémon 11...
Fetching Pokémon 12...
Fetching Pokémon 13...
Fetching Pokémon 14...
Fetching Pokémon 15...
Fetching Pokémon 16...
Fetching Pokémon 17...
Fetching Pokémon 18...
Fetching Pokémon 19...
Fetching Pokémon 20...
Fetching Pokémon 21...
Fetching Pokémon 22...
Fetching Pokémon 23...
Fetching Pokémon 24...
Fetching Pokémon 25...
Fetching Pokémon 26...
Fetching Pokémon 27...
Fetching Pokémon 28...
Fetching Pokémon 29...
Fetching Pokémon 30...
Fetching Pokémon 31...
Fetching Pokémon 32...
Fetching Pokémon 33...
Fetching Pokémon 34...
Fetching Pokémon 35...
Fetching Pokémon 36...
Fetching Pokémon 37...
Fetching Pokémon 38...
Fetching Pokémon 39...
Fetching Pokémon 40...
Fetching Pokémon 41...
Fetching Pokémon 42...
Fetching Pokémon 43...
Fetching Pokémon 44.

In [6]:
import csv
import json
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

DATA_FILE = Path("../data/pokemon151.json").resolve()
CSV_FILE = DATA_FILE.parent / "pokemon151.csv"
POKEAPI_TIMEOUT = 10
RETRY_DELAY = 2
MAX_RETRIES = 3
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept": "application/json",
}


def fetch_json(url: str) -> dict:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            request = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(request, timeout=POKEAPI_TIMEOUT) as response:
                return json.load(response)
        except urllib.error.HTTPError as e:
            print(f"HTTPError fetching {url}: {e.code} {e.reason}")
            if 500 <= e.code < 600 and attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
                continue
            raise
        except urllib.error.URLError as e:
            print(f"URLError fetching {url}: {e.reason}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
                continue
            raise


def parse_generation(generation_name: str) -> str:
    roman_map = {
        "i": "1",
        "ii": "2",
        "iii": "3",
        "iv": "4",
        "v": "5",
        "vi": "6",
        "vii": "7",
        "viii": "8",
        "ix": "9",
        "x": "10",
    }
    roman = generation_name.split("-")[-1].lower()
    return roman_map.get(roman, roman)


def parse_generation_from_version(version_name: str) -> str:
    version_map = {
        "red-blue": 1,
        "yellow": 1,
        "gold-silver": 2,
        "crystal": 2,
        "ruby-sapphire": 3,
        "emerald": 3,
        "firered-leafgreen": 3,
        "diamond-pearl": 4,
        "platinum": 4,
        "heartgold-soulsilver": 4,
        "black-white": 5,
        "black-2-white-2": 5,
        "x-y": 6,
        "omega-ruby-alpha-sapphire": 6,
        "sun-moon": 7,
        "ultra-sun-ultra-moon": 7,
        "sword-shield": 8,
        "brilliant-diamond-shining-pearl": 8,
        "lets-go-pikachu-lets-go-eevee": 8,
        "legends-arceus": 8,
        "scarlet-violet": 9,
    }
    return str(version_map.get(version_name.lower(), 0))


def latest_generation_from_game_indices(game_indices: list[dict]) -> str:
    versions = [item.get("version", {}).get("name", "") for item in game_indices]
    generations = [int(parse_generation_from_version(name)) for name in versions if parse_generation_from_version(name).isdigit()]
    return str(max(generations)) if generations else ""


with open(DATA_FILE, encoding="utf-8") as fp:
    pokemon_list = json.load(fp)

species_urls = {pokemon["species"]["url"] for pokemon in pokemon_list}
print(f"Loading {len(species_urls)} species records...")
species_cache = {}
for url in sorted(species_urls):
    species_cache[url] = fetch_json(url)
    time.sleep(0.05)

chain_urls = {species["evolution_chain"]["url"] for species in species_cache.values()}
print(f"Loading {len(chain_urls)} evolution chains...")
evolution_chain_cache = {}
for url in sorted(chain_urls):
    evolution_chain_cache[url] = fetch_json(url)
    time.sleep(0.05)

name_to_id = {pokemon["name"]: pokemon["id"] for pokemon in pokemon_list}
chain_info_map = {}

for url, chain in evolution_chain_cache.items():
    chain_id = int(urllib.parse.urlparse(url).path.rstrip("/").split("/")[-1])
    nodes = []

    def traverse(node, stage, parent_name):
        species_name = node["species"]["name"]
        child_names = [child["species"]["name"] for child in node["evolves_to"]]
        nodes.append({
            "species_name": species_name,
            "stage": stage,
            "pre": parent_name,
            "post": child_names,
        })
        for child in node["evolves_to"]:
            traverse(child, stage + 1, species_name)

    traverse(chain["chain"], 1, None)
    max_stage = max(node["stage"] for node in nodes)
    for node in nodes:
        chain_info_map[node["species_name"]] = {
            "evolution_id": chain_id,
            "evolution_stage": node["stage"],
            "remaining_evolutions": max_stage - node["stage"],
            "pre_species": node["pre"],
            "post_species": node["post"],
        }

rows = []
for pokemon in pokemon_list:
    species = species_cache[pokemon["species"]["url"]]
    species_names = {item["language"]["name"]: item["name"] for item in species["names"]}
    name_en = species_names.get("en", pokemon["name"].title())
    name_cn = (
        species_names.get("zh-Hans")
        or species_names.get("zh")
        or species_names.get("zh-Hant")
        or pokemon["name"].title()
    )

    types = sorted(pokemon["types"], key=lambda item: item["slot"])
    type1 = types[0]["type"]["name"] if len(types) > 0 else ""
    type2 = types[1]["type"]["name"] if len(types) > 1 else ""

    stats = {item["stat"]["name"]: item["base_stat"] for item in pokemon["stats"]}
    chain_info = chain_info_map.get(species["name"], {})
    pre_evolution_id = (
        name_to_id.get(chain_info.get("pre_species"))
        if chain_info.get("pre_species")
        else ""
    )
    post_ids = [
        name_to_id[name]
        for name in chain_info.get("post_species", [])
        if name in name_to_id
    ]
    post_evolution_id = ",".join(str(pokemon_id) for pokemon_id in post_ids)

    rows.append(
        {
            "pokemon_id": pokemon["id"],
            "name_en": name_en,
            "name_cn": name_cn,
            "type1": type1,
            "type2": type2,
            "hp": stats.get("hp", ""),
            "attack": stats.get("attack", ""),
            "defense": stats.get("defense", ""),
            "sp_attack": stats.get("special-attack", ""),
            "sp_defense": stats.get("special-defense", ""),
            "speed": stats.get("speed", ""),
            "evolution_id": chain_info.get("evolution_id", ""),
            "evolution_stage": chain_info.get("evolution_stage", ""),
            "remaining_evolutions": chain_info.get("remaining_evolutions", ""),
            "pre_evolution_id": pre_evolution_id,
            "post_evolution_id": post_evolution_id,
            "latest_generation": latest_generation_from_game_indices(pokemon.get("game_indices", [])),
        }
    )

fieldnames = [
    "pokemon_id",
    "name_en",
    "name_cn",
    "type1",
    "type2",
    "hp",
    "attack",
    "defense",
    "sp_attack",
    "sp_defense",
    "speed",
    "evolution_id",
    "evolution_stage",
    "remaining_evolutions",
    "pre_evolution_id",
    "post_evolution_id",
    "latest_generation",
]

with open(CSV_FILE, "w", encoding="utf-8", newline="") as fp:
    writer = csv.DictWriter(fp, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"完成：已保存 {len(rows)} 条记录到 {CSV_FILE}")

Loading 151 species records...


KeyboardInterrupt: 